# Fechamento do DLM — especificação e inferência

DLM finito irrestrito, TWFE UF/mês-ano, Δlog(I)/Δlog(1+D), contemporâneo e K_AIC. Inferência DK por regra de T + Bonferroni entre K + Holm entre exposições. Células executadas em processo Python compartilhado, sem kernel Jupyter; conferem tabelas externas e não alegam reestimação integral no notebook.

In [1]:
from pathlib import Path
import sys,json,hashlib,pandas as pd,numpy as np
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(root/'src'))
from dlm_inferencia_banda_regra_v3 import OUT,bandwidth
r=pd.read_csv(OUT/'resultados.csv')
s=pd.read_csv(root/'outputs/tables/dlm_fechamento_v4/resumo_diagnosticos.csv')
print('Regra L(T): pré',bandwidth(60),'total',bandwidth(119))


Regra L(T): pré 3 total 4


## Por que manter este DLM?

A monografia prevê efeitos fixos de unidades e tempo; o painel UF×mês preserva variação estadual. Não impomos Almon, pois VIF mediano dos máximos é 1,25, sem colinearidade extrema generalizada; VIF máximo 6,33 é ressalva localizada. K por AIC permanece escolhido em amostra comum. AIC gaussiano de trabalho não garante horizonte físico ou previsão ótima sob erros dependentes.

In [2]:
print(s[s.exposicao.eq('total_climatico')][['periodo','T','rho_abs_media','LB12_nominal_UFs','LB12_Holm_UFs','max_VIF']].to_string(index=False))
print('VIF máximos por aplicação: mediana',s.max_VIF.median(),'máximo',s.max_VIF.max())


periodo   T  rho_abs_media  LB12_nominal_UFs  LB12_Holm_UFs  max_VIF
    Pré  60       0.191362                13              7 1.196611
  Total 119       0.185201                20             16 1.165458
VIF máximos por aplicação: mediana 1.2509928905691488 máximo 6.330807941157472


## A calibração adicional foi rejeitada

Uma tentativa de testar dependência transversal reaplicando TWFE apresentou rejeição excessiva sob nulidade conhecida com UFs independentes e AR1=0,5. Seus p dos dados reais foram retirados. A auditoria preservada não comprova dependência; suas simulações mostram por que não pode ser usada. Correlação descritiva e CD ingênuo também não certificam dependência/independência. Escolha DK por robustez, sem preteste que determine a covariância.

In [3]:
a=pd.read_csv(root/'outputs/tables/dlm_fechamento_v4/auditoria_calibracao_rejeitada.csv')
print(a[['T','rho_AR1','replicacoes','taxa_rejeicao_nulo']].to_string(index=False))
assert a[a.rho_AR1.eq(.5)].taxa_rejeicao_nulo.min()>.5
print('Diagnóstico rejeitado: p não utilizados na inferência final.')


  T  rho_AR1  replicacoes  taxa_rejeicao_nulo
 60      0.0           50                0.04
 60      0.5           50                0.74
119      0.0           50                0.06
119      0.5           50                0.60
Diagnóstico rejeitado: p não utilizados na inferência final.


## Inferência final e leitura

DK/Bartlett com L(T) não exige independência entre UFs, mas depende de regularidade temporal e aproximações. Não corrige fatores omitidos/endogeneidade/raiz unitária. Seleção protegida depende de validade pontual uniforme; não é seletiva exata. Sensibilidades DK6/DK12/CR1 preservadas, sem escolher pelo menor p. Não rejeitar não comprova associação zero.

In [4]:
z=r[r.covariancia.eq('DK_regra') & r.status.eq('calculado')]
for per,g in z.groupby('periodo'):
 print(per,'rejeições após seleção + Holm:',int(g.p_Holm_familia.lt(.05).sum()))
assert not z.p_Holm_familia.lt(.05).any()
print(z[z.exposicao.eq('total_climatico')][['periodo','K','L','endpoint','estimate','p_selecao','p_Holm_familia']].to_string(index=False))


Pré rejeições após seleção + Holm: 0
Total rejeições após seleção + Holm: 0
periodo   K   L   endpoint  estimate  p_selecao  p_Holm_familia
    Pré 1.0 3.0      beta0  0.000139        1.0             1.0
    Pré 1.0 3.0     soma0K -0.000063        1.0             1.0
    Pré 1.0 3.0     soma1K -0.000201        1.0             1.0
    Pré 1.0 3.0 conjunto0K  0.362172        1.0             1.0
    Pré 1.0 3.0 conjunto1K  0.330724        1.0             1.0
  Total 1.0 4.0      beta0  0.000623        1.0             1.0
  Total 1.0 4.0     soma0K  0.001146        1.0             1.0
  Total 1.0 4.0     soma1K  0.000524        1.0             1.0
  Total 1.0 4.0 conjunto0K  1.869228        1.0             1.0
  Total 1.0 4.0 conjunto1K  1.427141        1.0             1.0


## Decisão consolidada

Ajuste técnico desta etapa encerrado no escopo descrito; análise permanece observacional exploratória. Granger não reexecutado; SARIMAX não iniciado. Fontes da decisão, fórmulas e limites em `docs/dlm_fechamento_v4_decisao.md`. Estimação/inferência em scripts v3; tentativa diagnóstica rejeitada arquivada separadamente.